# OMOP Provider and Care Site Tables

Transforms FHIR Practitioner and Organization resources into OMOP CDM `provider` and `care_site` tables.

## Mapping: FHIR Practitioner → OMOP Provider

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| provider_id | Practitioner.id | Hash to integer |
| provider_name | Practitioner.name | Formatted name |
| npi | Practitioner.identifier[NPI] | NPI identifier |
| specialty_concept_id | Practitioner.qualification | Map to specialty |
| care_site_id | Practitioner.practitionerRole | Reference to care_site |
| provider_source_value | Practitioner.id | Original ID |

## Mapping: FHIR Organization → OMOP Care_Site

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| care_site_id | Organization.id | Hash to integer |
| care_site_name | Organization.name | Organization name |
| place_of_service_concept_id | Organization.type | Map to place of service |
| location_id | Organization.address | Reference to location |
| care_site_source_value | Organization.id | Original ID |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Care Site Streaming Table (from Organization)

In [ ]:
DECLARE OR REPLACE VARIABLE create_care_site_stmt STRING;

SET VARIABLE create_care_site_stmt = "
CREATE OR REFRESH STREAMING TABLE care_site (
  -- Primary key
  care_site_id BIGINT NOT NULL COMMENT 'Unique care site identifier'
  
  -- Name
  ,care_site_name STRING COMMENT 'Organization name'
  
  -- Place of service
  ,place_of_service_concept_id INT DEFAULT 0 COMMENT 'Place of service concept'
  
  -- Location reference
  ,location_id BIGINT COMMENT 'Reference to location table'
  
  -- Source values
  ,care_site_source_value STRING COMMENT 'Original Organization ID'
  ,place_of_service_source_value STRING COMMENT 'Original organization type'
  
  -- Address info
  ,address_line STRING COMMENT 'Street address'
  ,city STRING COMMENT 'City'
  ,state STRING COMMENT 'State'
  ,postal_code STRING COMMENT 'Postal code'
  
  -- Lineage
  ,fhir_organization_uuid STRING COMMENT 'Original FHIR Organization UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Care Site table - Organizations from FHIR Organization resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer care_site_id
  ABS(HASH(COALESCE(id::STRING, organization_uuid))) AS care_site_id
  
  -- Name
  ,name::STRING AS care_site_name
  
  -- Place of service (map organization type)
  ,CASE type[0]:coding[0]:code::STRING
    WHEN 'prov' THEN 9201    -- Provider's office
    WHEN 'hosp' THEN 8717    -- Inpatient hospital
    WHEN 'crs' THEN 8883     -- Skilled nursing
    WHEN 'pharm' THEN 8764   -- Pharmacy
    WHEN 'lab' THEN 8782     -- Lab
    ELSE 0
  END AS place_of_service_concept_id
  
  -- Location (to be linked)
  ,NULL AS location_id
  
  -- Source values
  ,id::STRING AS care_site_source_value
  ,type[0]:coding[0]:code::STRING AS place_of_service_source_value
  
  -- Address
  ,address[0]:line[0]::STRING AS address_line
  ,address[0]:city::STRING AS city
  ,address[0]:state::STRING AS state
  ,address[0]:postalCode::STRING AS postal_code
  
  -- Lineage
  ,organization_uuid AS fhir_organization_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".organization)
WHERE active::STRING = 'true' OR active IS NULL
";

SELECT create_care_site_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_care_site_stmt;

## Create Provider Streaming Table (from Practitioner)

In [ ]:
DECLARE OR REPLACE VARIABLE create_provider_stmt STRING;

SET VARIABLE create_provider_stmt = "
CREATE OR REFRESH STREAMING TABLE provider (
  -- Primary key
  provider_id BIGINT NOT NULL COMMENT 'Unique provider identifier'
  
  -- Name
  ,provider_name STRING COMMENT 'Provider name'
  
  -- NPI
  ,npi STRING COMMENT 'National Provider Identifier'
  ,dea STRING COMMENT 'DEA number'
  
  -- Specialty
  ,specialty_concept_id INT DEFAULT 0 COMMENT 'Specialty concept'
  
  -- Care site reference
  ,care_site_id BIGINT COMMENT 'Reference to care_site table'
  
  -- Gender
  ,gender_concept_id INT DEFAULT 0 COMMENT 'Gender concept'
  
  -- Birth year
  ,year_of_birth INT COMMENT 'Year of birth'
  
  -- Source values
  ,provider_source_value STRING COMMENT 'Original Practitioner ID'
  ,specialty_source_value STRING COMMENT 'Original specialty value'
  ,specialty_source_concept_id INT DEFAULT 0 COMMENT 'Source specialty concept'
  ,gender_source_value STRING COMMENT 'Original gender value'
  ,gender_source_concept_id INT DEFAULT 0 COMMENT 'Source gender concept'
  
  -- Lineage
  ,fhir_practitioner_uuid STRING COMMENT 'Original FHIR Practitioner UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Provider table - Practitioners from FHIR Practitioner resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer provider_id
  ABS(HASH(COALESCE(id::STRING, practitioner_uuid))) AS provider_id
  
  -- Format name
  ,CONCAT_WS(' ',
    name[0]:given[0]::STRING,
    name[0]:family::STRING
  ) AS provider_name
  
  -- NPI (look for NPI system in identifiers)
  ,(
    SELECT MAX(i.value::STRING)
    FROM LATERAL variant_explode(identifier) AS i
    WHERE i.value:system::STRING LIKE '%npi%'
  ) AS npi
  ,NULL AS dea
  
  -- Specialty (from qualification)
  ,0 AS specialty_concept_id
  
  -- Care site (would need PractitionerRole to link)
  ,NULL AS care_site_id
  
  -- Gender
  ,CASE LOWER(gender::STRING)
    WHEN 'male' THEN 8507
    WHEN 'female' THEN 8532
    ELSE 0
  END AS gender_concept_id
  
  -- Birth year
  ,NULL AS year_of_birth
  
  -- Source values
  ,id::STRING AS provider_source_value
  ,qualification[0]:code:coding[0]:display::STRING AS specialty_source_value
  ,0 AS specialty_source_concept_id
  ,gender::STRING AS gender_source_value
  ,0 AS gender_source_concept_id
  
  -- Lineage
  ,practitioner_uuid AS fhir_practitioner_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".practitioner)
WHERE active::STRING = 'true' OR active IS NULL
";

SELECT create_provider_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_provider_stmt;

In [ ]:
-- Verify care_site table
SELECT 
  care_site_id,
  care_site_name,
  city,
  state,
  care_site_source_value
FROM care_site
LIMIT 10;

In [ ]:
-- Verify provider table
SELECT 
  provider_id,
  provider_name,
  npi,
  specialty_source_value,
  provider_source_value
FROM provider
LIMIT 10;

In [ ]:
-- Summary statistics
SELECT 'care_site' AS table_name, COUNT(*) AS row_count FROM care_site
UNION ALL
SELECT 'provider' AS table_name, COUNT(*) AS row_count FROM provider;